In [8]:
import pandas as pd
import sqlite3
import os
import numpy as np

# --- PHASE 1: INITIALISIERUNG (Einmalig) ---
# Wir prüfen, ob die DB schon existiert, um Zeit zu sparen
if not os.path.exists('apple_apps.db'):
    print("Initialisiere Datenbank aus CSV...")
    
    # 1. Rohdaten laden
    df = pd.read_csv('appleAppData.csv')
    df.columns = df.columns.str.strip()
    
    # 2. Preprocessing
    # Nur bewertete Apps und Typ-Kategorisierung
    if 'Size_Bytes' in df.columns:
        df['Size_MB'] = df['Size_Bytes'] / (1024 * 1024)
    dashboard_df = df[df['Average_User_Rating'] > 0].copy()
    dashboard_df['Type'] = dashboard_df['Free'].apply(lambda x: 'Free' if x else 'Paid')
    
    # 3. In SQL speichern
    conn = sqlite3.connect('apple_apps.db')
    dashboard_df.to_sql('apps', conn, if_exists='replace', index=False)
    conn.close()
    print("Datenbank 'apple_apps.db' erfolgreich erstellt.")

    # JETZT DEN SPEICHER LEEREN
    import gc # Garbage Collector

    if 'df' in locals():
        del df
    
    if 'dashboard_df' in locals():
        del dashboard_df

    gc.collect() # Zwingt Python, den Speicher jetzt sofort freizugeben
   # print("CSV-Daten aus dem RAM gelöscht. Wir arbeiten jetzt nur noch mit SQL.")
    
else:
    print("Datenbank bereits vorhanden. Springe direkt zur Analyse.")

# --- PHASE 2: AB JETZT NUR NOCH SQL ---
def get_data(query):
    conn = sqlite3.connect('apple_apps.db')
    result = pd.read_sql(query, conn)
    conn.close()
    return result

# Beispiel: Den "neuen" dashboard_df direkt aus der DB ziehen
dashboard_df = get_data("SELECT * FROM apps")
print(f"Dataset ready from SQL with {len(dashboard_df)} rated apps.")

Initialisiere Datenbank aus CSV...
Datenbank 'apple_apps.db' erfolgreich erstellt.
Dataset ready from SQL with 546056 rated apps.


In [ ]:
# 1. SQL-Abfrage für das Premium-Segment
# Wir filtern auf Price > 0 und aggregieren direkt in der DB
query_price = """
SELECT 
    Primary_Genre, 
    AVG(Price) as Avg_Price, 
    COUNT(App_Name) as App_Count
FROM apps
WHERE Price > 0
GROUP BY Primary_Genre
ORDER BY Avg_Price DESC
LIMIT 15
"""

price_analysis = get_data(query_price)

# 2. Visualisierung
fig_price = px.bar(
    price_analysis, 
    x='Avg_Price', 
    y='Primary_Genre', 
    orientation='h',
    color='Avg_Price',
    color_continuous_scale='Blues',
    labels={'Avg_Price': 'Average Price ($)', 'Primary_Genre': 'Category'},
    title='Top 15 Most Expensive App Categories (Premium Segment)'
)

fig_price.update_layout(
    yaxis={'categoryorder':'total ascending'},
    template='plotly_dark'
)
fig_price.show()

print(f"Preisanalyse für {len(price_analysis)} Top-Genres abgeschlossen.")

Preisanalyse für 15 Top-Genres abgeschlossen.


Die Analyse räumt mit dem Mythos auf, dass Erfolg im App Store käuflich oder durch reine Masse (Reviews) garantiert ist. Da selbst das Rating nur einen moderaten Einfluss hat, wird klar: Der wahre 'Impact' entsteht durch ein komplexes Zusammenspiel, bei dem die Qualität (Rating) die Basis bildet, aber erst durch die Reichweite (Reviews) skaliert wird

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

def get_data(query):
    conn = sqlite3.connect('apple_apps.db')
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

st.set_page_config(page_title="Chris' App Insights", layout="wide")
st.title("App Store Analytics Dashboard")

# --- SIDEBAR ---
st.sidebar.header("Filter & Steuerung")
min_rating = st.sidebar.slider("Mindest-Rating", 0.0, 5.0, 4.0)
min_reviews = st.sidebar.number_input("Mindestanzahl Reviews", 0, 1000000, 1000)

# --- DATEN HOLEN ---
query = f"""
SELECT App_Name, Average_User_Rating, Reviews, Price, Size_MB, Primary_Genre 
FROM apps 
WHERE Average_User_Rating >= {min_rating} AND Reviews >= {min_reviews}
"""
df = get_data(query)
df['Size_MB'] = pd.to_numeric(df['Size_MB'], errors='coerce').fillna(0)
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce').fillna(0)

# --- MODULE 1: METRIKEN ---
col1, col2, col3 = st.columns(3)
col1.metric("Apps im Fokus", len(df))
col2.metric("Ø Rating", f"{df['Average_User_Rating'].mean():.2f}")
col3.metric("Ø Preis", f"{df['Price'].mean():.2f} €")

# --- MODULE 2: DER CLUSTER (SCATTER) ---
st.subheader("Markt-Übersicht: Rating vs. Popularität")
fig_scatter = px.scatter(
    df, x="Reviews", y="Average_User_Rating", 
    size="Size_MB", color="Primary_Genre",
    hover_name="App_Name", log_x=True,
    template="plotly_dark", size_max=30
)
st.plotly_chart(fig_scatter, use_container_width=True)

# --- MODULE 3: DIE TRUE GIANTS (Top Apps nach Reviews) ---
st.subheader("🏆 Die True Giants (Meistbewertet)")
top_apps = df.nlargest(10, 'Reviews')[['App_Name', 'Reviews', 'Average_User_Rating']]
st.table(top_apps)

# --- MODULE 4: GENRE HEATMAP (Verteilung) ---
st.subheader("📊 Genre-Verteilung (Heatmap)")
genre_counts = df['Primary_Genre'].value_counts().reset_index()
genre_counts.columns = ['Genre', 'Anzahl']
fig_pie = px.pie(genre_counts, values='Anzahl', names='Genre', hole=0.4, template="plotly_dark")
st.plotly_chart(fig_pie, use_container_width=True)

#st.info("💡 Alle Daten werden live aus deiner SQL-Datenbank gefiltert.")

Overwriting app.py


In [ ]:
#streamlit run app.py 

SyntaxError: invalid syntax (909855155.py, line 1)